# A3 corregido: Red híbrida CNN-LSTM (correcciones i y iii)

Cambios respecto a `A3_Red Híbrida CNN-LSTM.ipynb` (el original queda intacto):
1. La etiqueta es `was_preterm` (antes `was_early_preterm`), igual que en A1 y A2.
2. La prueba ya no es una sola partición estratificada por centro (`project`):
   se usan los pliegues comunes por paciente (`pliegues_por_paciente.csv`,
   5 pliegues x 5 repeticiones, estratificados por la etiqueta).
3. Se guardan las probabilidades por paciente de cada pliegue en
   `resultados/pred_A3.csv` para armar la Tabla 4.1.

Punto iii (entrenamiento y umbral):

4. La validación (25 % de las pacientes de entrenamiento) se separa **estratificada por la etiqueta**.
5. El punto de control (`ModelCheckpoint`) guarda los pesos con menor `val_loss`
   (antes `val_accuracy`), y se agrega `EarlyStopping` sobre `val_loss` (paciencia 50 épocas,
   máximo 500).
6. Se agrega ponderación de clases, w_c = N / (2·N_c), calculada con el entrenamiento de cada pliegue.
7. El umbral de decisión es el índice de Youden, max(TPR − FPR), calculado con las predicciones
   del conjunto de **validación**. Se congela y se aplica una sola vez al pliegue de prueba.
8. Al final: matriz de confusión por paciente (repetición 0) y la figura de selección del umbral
   sobre validación, para el Capítulo 4.

9. Bloque denso: se agregan las tres capas `Dense(100)` que describe la Sec. 3.5.3, cada una
   seguida de LeakyReLU (pendiente 0.01), BatchNormalization y Dropout(0.15). El original tenía
   `LeakyReLU(100)` sin capas densas, lo que ponía la pendiente de los negativos en 100.

La Conv1D, la LSTM y los datos de entrada **no** cambian.

`cst_valencia.csv` no está en el repositorio. Sus 16 columnas (CST, subCST, score y las
13 similitudes `*_sim`) están dentro de `cst_valencia_w_taxons.csv`, así que se toman de
ahí. Con eso se reproduce la tabla original: 1895 x 10158 y 10138 predictores.

**Google Colab:** sube la carpeta `CorreccionesTesis` (CSV, `comun.py`, `pliegues_por_paciente.csv`
y este notebook) a *Mi unidad* de Google Drive, activa la GPU (Entorno de ejecución → Cambiar
tipo de entorno → GPU) y corre todas las celdas. La primera celda monta Drive y entra a la carpeta.
En tu computadora esa celda no hace nada.

In [ ]:
# Solo en Google Colab: monta Google Drive y entra a la carpeta con los datos y comun.py.
# En una computadora local esta celda no hace nada.
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CARPETA = '/content/drive/MyDrive/CorreccionesTesis'   # cámbiala si la subiste a otro lugar
    os.chdir(CARPETA)
    print('Trabajando en', os.getcwd())
except ImportError:
    print('No es Colab: se usa la carpeta actual,', os.getcwd())

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import os, sys
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import random
import numpy as np
import pandas as pd
import tensorflow as tf
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
tf.config.experimental.enable_op_determinism()

from tensorflow.keras.preprocessing import sequence
from tensorflow.keras import backend as K
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, confusion_matrix
import seaborn as sns

sys.path.insert(0, os.path.abspath('.'))
from comun import (leer_datos, RESULTADOS, ETIQUETA, cargar_pliegues, iterar_pliegues,
                   umbral_youden, metricas)
RESULTADOS.mkdir(exist_ok=True)
EPOCHS = int(os.environ.get('A3_EPOCHS', 500))
PACIENCIA = 50          # EarlyStopping: épocas sin mejorar val_loss antes de parar
print('Keras', keras.__version__, '| épocas:', EPOCHS)

In [ ]:
community = leer_datos('cst_valencia_w_taxons')
cols_cst = ['specimen', 'CST', 'subCST', 'score'] + [c for c in community.columns if c.endswith('_sim')]
alpha_diversity = community[cols_cst].copy()   # equivale a cst_valencia.csv del original
metadata = leer_datos('metadata')
phylotypes = leer_datos('phylotype_nreads.1e_1')

metadata.loc[metadata.age=='Unknown','age']='99'
CatVal = ["Below_18","Above_38","99","29-38","18_to_28"]
metadata_num = metadata.loc[~metadata["age"].isin(CatVal)]
metadata_num = metadata_num.astype({"age": float})
metadata_num.age = metadata_num.age.round()
metadata_num = metadata_num.astype({"age": int})
bins = pd.IntervalIndex.from_tuples([(0, 18), (18, 28), (28, 38),(38,80)])
metadata_num["age"] = pd.cut(metadata_num["age"], bins)
metadata_num['age'] = metadata_num['age'].astype(str)
metadata_num.loc[metadata_num.age=='(0, 18]','age']="Below_18"
metadata_num.loc[metadata_num.age=='(18, 28]','age']="18_to_28"
metadata_num.loc[metadata_num.age=='(28, 38]','age']="29-38"
metadata_num.loc[metadata_num.age=='(38, 80]','age']="Above_38"
metadata_car = metadata.loc[metadata["age"].isin(CatVal)]
metadata = pd.concat([metadata_num, metadata_car], axis=0, ignore_index=True)

combineddata = alpha_diversity.merge(community, left_on='specimen', right_on='specimen')
combineddata = combineddata.merge(metadata, left_on='specimen', right_on='specimen')
combineddata = combineddata.merge(phylotypes, left_on='specimen', right_on='specimen')
X = combineddata
print(X.shape)

In [ ]:
Cols1 = X.columns[1:422]
Cols2 = ["collect_wk"]
Cols3 = X.columns[435:len(X.columns)]
ColstoGet = np.append(Cols1,Cols2)
ColstoGet = np.append(ColstoGet,Cols3)
for c in ['index_y', 'CST', 'subCST', 'NIH Ethnicity Category', 'NIH Racial Category', 'race',
          # Variables de desenlace: nunca deben ser predictores (son booleanas y el filtro
          # numérico ya las quitaba en el original; se excluyen de forma explícita).
          'delivery_wk', 'was_term', 'was_preterm', 'was_early_preterm']:
    ColstoGet = ColstoGet[ColstoGet != c]

Predictors = np.append(ColstoGet, 'participant_id')
Predictors = np.append(Predictors, [ETIQUETA, 'project'])   # CAMBIO: was_preterm
print(Predictors.shape)

In [ ]:
X_ids = combineddata["specimen"]
X = X.loc[:, Predictors]
X["specimen_person"] = X_ids.str.split('-').str[0]
X = X.loc[:, X.dtypes.apply(lambda dt: np.issubdtype(dt, np.number)) | (X.columns == "specimen_person")]
for col in [ETIQUETA, 'participant_id']:                   # CAMBIO: was_preterm
    if col not in X.columns:
        X[col] = combineddata[col]

persons = X["specimen_person"].unique()
lstmDat_X, lstmDat_y = [], []
for person in persons:
    loopDat = X.loc[X["specimen_person"] == person]
    loopy = loopDat[ETIQUETA].unique()                     # CAMBIO: was_preterm
    loopDat = loopDat.drop(['specimen_person', ETIQUETA, 'participant_id'], axis=1)
    lstmDat_X.append(loopDat.to_numpy())
    lstmDat_y.append(loopy)

lstmDat_y_flat = [item for sublist in lstmDat_y for item in sublist]
lstmDat_y_flat = np.where(np.array(lstmDat_y_flat) == True, 1, 0)

max_sequence_length = 9
X_padded = sequence.pad_sequences(lstmDat_X, maxlen=max_sequence_length)
n_features = X_padded.shape[2]
print("X_padded shape:", X_padded.shape, "| pretérmino:", lstmDat_y_flat.sum(), "de", len(lstmDat_y_flat))
assert n_features == 10138

## Pliegues comunes por paciente
Antes: `train_test_split(..., stratify=project)`, una sola partición. Ahora: los mismos
25 pliegues por paciente que usan la regresión logística y el Random Forest.

In [ ]:
pliegues = cargar_pliegues()
indice = {p: i for i, p in enumerate(persons)}
assert set(indice) == set(pliegues.participant_id)

def construir_modelo():
    K.clear_session()
    x = layers.Input(shape=(max_sequence_length, n_features))
    model = layers.Conv1D(filters=16, kernel_size=3, padding='same', activation='elu')(x)
    model = layers.MaxPooling1D(pool_size=2)(model)
    model = layers.LSTM(100)(model)
    for _ in range(3):
        # Capa densa de 100 neuronas + LeakyReLU con pendiente 0.01 (punto iii; Sec. 3.5.3).
        # El original usaba LeakyReLU(100) sin capa densa: eso fijaba la pendiente en 100.
        model = layers.Dense(100)(model)
        model = layers.LeakyReLU(0.01)(model)
        model = layers.BatchNormalization()(model)
        model = layers.Dropout(0.15)(model)
    model = layers.Dense(1, activation='sigmoid')(model)
    myModel = keras.Model(x, model)
    myModel.compile(optimizer=Adam(learning_rate=1e-3), loss='binary_crossentropy', metrics=['accuracy'])
    return myModel

construir_modelo().summary()

In [ ]:
predicciones, pred_val, filas, historias = [], [], [], {}
for rep, fold, ids_tr, ids_te in iterar_pliegues(pliegues):
    tr = np.array(sorted(indice[p] for p in ids_tr))
    te = np.array(sorted(indice[p] for p in ids_te))
    X_train, X_test = X_padded[tr], X_padded[te]
    y_train, y_test = lstmDat_y_flat[tr], lstmDat_y_flat[te]
    # Validación: 25 % de las pacientes de entrenamiento, estratificada por la etiqueta (punto iii)
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25,
                                                      stratify=y_train, random_state=SEED)
    # Ponderación de clases w_c = N / (2 N_c), con el entrenamiento del pliegue
    n, n1 = len(y_train), int(y_train.sum())
    pesos = {0: n / (2 * (n - n1)), 1: n / (2 * n1)}

    keras.utils.set_random_seed(SEED)
    myModel = construir_modelo()
    checkpoint_filepath = f'./best_model_r{rep}_f{fold}.weights.h5'
    cb = [tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_filepath, save_weights_only=True,
                                             monitor='val_loss', mode='min', save_best_only=True),
          tf.keras.callbacks.EarlyStopping(monitor='val_loss', mode='min', patience=PACIENCIA)]
    h = myModel.fit(X_train, y_train, epochs=EPOCHS, validation_data=(X_val, y_val),
                    batch_size=32, callbacks=cb, class_weight=pesos, verbose=0)
    historias[(rep, fold)] = h.history
    myModel.load_weights(checkpoint_filepath)
    os.remove(checkpoint_filepath)

    # Umbral de Youden con la VALIDACIÓN; luego se aplica una sola vez a prueba (punto iii)
    yval_pred = myModel.predict(X_val, verbose=0).ravel()
    umbral = umbral_youden(y_val, yval_pred)
    ypred = myModel.predict(X_test, verbose=0).ravel()
    m = metricas(y_test, ypred, umbral)
    mejor_epoca = int(np.argmin(h.history['val_loss'])) + 1
    filas.append(dict(modelo='CNN-LSTM', rep=rep, fold=fold, **m,
                      epocas_corridas=len(h.history['loss']), mejor_epoca=mejor_epoca))
    pred_val.append(pd.DataFrame({'rep': rep, 'fold': fold, 'y': y_val, 'prob': yval_pred}))
    predicciones.append(pd.DataFrame({'rep': rep, 'fold': fold, 'participant_id': persons[te],
                                      'y': y_test, 'prob': ypred}))
    print(f"rep {rep} fold {fold}: AUC-ROC {m['auc_roc']:.3f}  AUC-PR {m['auc_pr']:.3f}  "
          f"recall {m['recall']:.2f}  umbral {umbral:.3f}  épocas {len(h.history['loss'])} "
          f"(mejor {mejor_epoca})", flush=True)

pd.concat(predicciones).to_csv(RESULTADOS / 'pred_A3.csv', index=False)
pd.concat(pred_val).to_csv(RESULTADOS / 'pred_val_A3.csv', index=False)
res = pd.DataFrame(filas)
res.to_csv(RESULTADOS / 'metricas_A3.csv', index=False)
res[['auc_roc', 'auc_pr', 'accuracy', 'precision', 'recall', 'f1', 'umbral']].agg(['mean', 'std']).round(3)

In [ ]:
# Curvas de pérdida del primer pliegue (antes se dibujaban en vivo con PlotLosses)
h = historias[(0, 0)]
plt.plot(h['loss'], label='loss'); plt.plot(h['val_loss'], label='v_loss')
plt.xlabel('Épocas'); plt.ylabel('Pérdida (Loss)'); plt.title('Evolución de la pérdida (rep 0, pliegue 0)')
plt.legend(); plt.savefig(RESULTADOS / 'LossEpocas_A3.png', dpi=200); plt.show()

## Matriz de confusión por paciente (repetición 0)
En la repetición 0 cada una de las 616 pacientes cae en prueba exactamente una vez. Cada paciente se
clasifica con el umbral de su pliegue, calculado con la validación de ese pliegue.

In [ ]:
pred = pd.concat(predicciones)
umbrales = res.set_index(['rep', 'fold']).umbral
r0 = pred[pred.rep == 0].copy()
r0['pred'] = (r0.prob >= r0.apply(lambda f: umbrales[(f.rep, f.fold)], axis=1)).astype(int)
cm = confusion_matrix(r0.y, r0.pred, labels=[0, 1])
pd.DataFrame(cm, index=['Real: término', 'Real: pretérmino'],
             columns=['Predicho: término', 'Predicho: pretérmino']).to_csv(RESULTADOS / 'matriz_confusion_A3.csv')
tn, fp, fn, tp = cm.ravel()
print(f'VN={tn}  FP={fp}  FN={fn}  VP={tp}  | recall={tp/(tp+fn):.3f}  especificidad={tn/(tn+fp):.3f}  '
      f'precisión={tp/(tp+fp):.3f}')
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Término', 'Pretérmino'], yticklabels=['Término', 'Pretérmino'])
plt.title('Matriz de confusión por paciente (CNN-LSTM)'); plt.ylabel('Real'); plt.xlabel('Predicho')
plt.tight_layout(); plt.savefig(RESULTADOS / 'MatrixConfusion_A3_paciente.png', dpi=200); plt.show()

## Selección del umbral sobre validación (repetición 0, pliegue 0)
Sensibilidad (TPR) y especificidad (1 − FPR) en función del umbral, calculadas con el conjunto de
validación. La línea vertical marca el umbral de Youden, el que maximiza TPR − FPR.

In [ ]:
v = pd.concat(pred_val)
v = v[(v.rep == 0) & (v.fold == 0)]
fpr, tpr, thr = roc_curve(v.y, v.prob)
thr = np.minimum(thr, 1.0)
u = umbrales[(0, 0)]
plt.figure(figsize=(7, 5))
plt.plot(thr, tpr, label='Sensibilidad (TPR)')
plt.plot(thr, 1 - fpr, color='red', label='Especificidad (1 - FPR)')
plt.axvline(u, color='k', ls='--', label=f'Umbral de Youden = {u:.3f}')
plt.xlabel('Umbral de decisión'); plt.ylabel('Tasa'); plt.title('Selección del umbral (validación, rep 0, pliegue 0)')
plt.legend(); plt.tight_layout(); plt.savefig(RESULTADOS / 'Umbral_validacion_A3.png', dpi=200); plt.show()
print(res[['umbral', 'epocas_corridas', 'mejor_epoca']].describe().round(3))